# **_----Baiseline для классических моделей ML---_**

**_Список моделей:_**
1. Линейная регрессия с регуляризацией:
    1. Lasso
    2. Ridge
    3. ElasticNet

2. KNN (sklearn) с разными гиперпараметрами (n_neighbors, weight, metric)
3. Решающее дерево (sklearn)
4. Random Forest (sklearn) с разными гиперпараметрами
5. Бустинги
    1. CatBoost
    2. LightGBM
    3. XGBoost

In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, Lasso, ElasticNet
import config
from preprocessing import preprocess_data
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

In [9]:
# Загрузка данных
df_train = pd.read_csv(config.TRAIN_PATH)
df_test = pd.read_csv(config.TEST_PATH)
test_ids = df_test[config.ID_COL]

# Предобработка
df_train_proc, df_test_proc = preprocess_data(df_train, df_test)
# Разделение на X и y
X_train = df_train_proc.drop(columns=[config.ID_COL, config.TARGET_COL])
y_train = np.log1p(df_train_proc[config.TARGET_COL])
X_test = df_test_proc.drop(columns=[config.ID_COL])
print(f"Готово! X_train: {X_train.shape}, y_train: {y_train.shape}")

Готово! X_train: (1460, 208), y_train: (1460,)


In [14]:
# Универсальная функция оценки для ВСЕХ моделей (линейных, деревьев, бустингов)
kf = KFold(n_splits=config.N_SPLITS, shuffle=config.SHUFFLE, random_state=config.RANDOM_STATE)
def evaluate_model(module_name, model, X, y, kf, use_scaler=True):
    """
    Принцип работы функции:
    - use_scaler=True -> добавляет StandardScaler (для линейных моделей и KNN)
    - use_scaler=False -> обучает напрямую без скейлера (для деревьев, леса и бустингов)
    """
    # Выбираем, нужен ли скейлер
    if use_scaler:
        estimator = Pipeline([
            ('scaler', StandardScaler()),
            ('model', model)
        ])
    else:
        estimator = model  # деревья и бустинги обучаем на исходных признаках

    fold_rmsl_errors = []
    oof_predictions = np.zeros(len(X))
    # 2. Цикл кросс-валидации
    for fold, (train_index, val_index) in enumerate(kf.split(X), 1):
        X_tr, y_tr = X.iloc[train_index], y.iloc[train_index]
        X_val, y_val = X.iloc[val_index], y.iloc[val_index]

        # Обучаем estimator (пайплайн со скейлером или чистую модель)
        estimator.fit(X_tr, y_tr)
        val_preds = estimator.predict(X_val)
        oof_predictions[val_index] = val_preds
        fold_rmsl_errors.append(root_mean_squared_error(y_val, val_preds))
    # 3. Расчет итоговых метрик
    mean_rmsle = np.mean(fold_rmsl_errors)
    std_rmsle = np.std(fold_rmsl_errors)
    real_dollars = np.expm1(y)
    pred_dollars = np.expm1(oof_predictions)
    mae = mean_absolute_error(real_dollars, pred_dollars)
    r2 = r2_score(y, oof_predictions)
    return {
        "Модель": module_name,
        "RMSLE": round(mean_rmsle, 4),
        "RMSLE (std)": round(std_rmsle, 4),
        "MAE ($)": f"${mae:,.2f}",
        "R^2": round(r2, 4)
    }

In [17]:
import importlib
importlib.reload(config)
results = []
# Обучаем модели со скейлером (Линейные + KNN)
for name, model in config.MODELS_SCALED.items():
    print(f"Обучаю со скейлером: {name}...")
    results.append(evaluate_model(name, model, X_train, y_train, kf, use_scaler=True))
# Обучаем модели без скейлера (Деревья, Лес, Бустинги)
for name, model in config.MODELS_UNSCALED.items():
    print(f"Обучаю без скейлера: {name}...")
    results.append(evaluate_model(name, model, X_train, y_train, kf, use_scaler=False))
# Выводим единый красивый Лидерборд:
df_results = pd.DataFrame(results).sort_values(by="RMSLE").reset_index(drop=True)
df_results

Обучаю со скейлером: Ridge (alpha=10)...
Обучаю со скейлером: Lasso (alpha=0.0005)...
Обучаю со скейлером: ElasticNet (alpha=0.001)...
Обучаю со скейлером: KNN (k=5, uniform)...
Обучаю со скейлером: KNN (k=5, distance)...
Обучаю со скейлером: KNN (k=10, distance)...
Обучаю со скейлером: KNN (k=10, manhattan)...
Обучаю без скейлера: Decision Tree (default)...
Обучаю без скейлера: Decision Tree (depth=6)...
Обучаю без скейлера: Random Forest (100 trees)...
Обучаю без скейлера: Random Forest (200 trees, sqrt)...
Обучаю без скейлера: Random Forest (200 trees, depth=15)...
Обучаю без скейлера: CatBoost (500 trees)...
Обучаю без скейлера: LightGBM (500 trees)...
Обучаю без скейлера: XGBoost (500 trees)...


,Модель,RMSLE,RMSLE (std),MAE ($),R^2
0,CatBoost (500 trees),0.1271,0.0177,"$15,367.67",0.8968
1,LightGBM (500 trees),0.1354,0.0176,"$16,555.49",0.8831
2,XGBoost (500 trees),0.1411,0.0195,"$16,691.18",0.8727
3,"Random Forest (200 trees, depth=15)",0.1436,0.0187,"$17,524.20",0.8686
4,"Random Forest (200 trees, sqrt)",0.1441,0.0149,"$17,425.86",0.8684
5,Random Forest (100 trees),0.1443,0.0186,"$17,622.60",0.8673
6,Ridge (alpha=10),0.1553,0.0326,"$16,800.99",0.8420
7,ElasticNet (alpha=0.001),0.1654,0.0346,"$16,596.32",0.8210
8,Lasso (alpha=0.0005),0.1669,0.0351,"$16,568.86",0.8175
9,"KNN (k=10, manhattan)",0.1835,0.0136,"$22,982.89",0.7877
